# Explore the data yourself

Every dataset in the project is available as a SQL table (DuckDB) and as a pandas DataFrame.
Run `make pipeline` first if `data/raw/` is empty. Select the **`.venv`** kernel (top right in VS Code).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # make `sei` importable from notebooks/

import pandas as pd
import plotly.express as px
from sei.sql import q, tables
from sei import config as C, stats as S

tables()[["table_name", "n_columns"]]

## 1. SQL: query anything
`q("...")` returns a pandas DataFrame. See `queries/examples.sql` for 14 more queries.

In [ ]:
q('''
SELECT arm,
       COUNT(*)                                 AS users,
       ROUND(AVG(exp_minutes_per_day), 2)       AS minutes_per_day,
       ROUND(100 * AVG(churned), 2)             AS churn_pct,
       ROUND(AVG(exp_ad_revenue_per_day)*365,2) AS annual_revenue_per_user
FROM user_experiment
GROUP BY arm ORDER BY arm
''')

## 2. Your own segment cut
Change the `WHERE` clause: e.g. only iOS users in Europe.

In [ ]:
df = q('''
SELECT arm, age_group, AVG(exp_minutes_per_day) AS minutes, AVG(churned) AS churn
FROM user_experiment
WHERE device = 'iOS' AND region = 'Europe'
GROUP BY ALL ORDER BY age_group, arm
''')
px.bar(df, x="age_group", y="churn", color="arm", barmode="group",
       color_discrete_map=C.ARM_COLORS, title="Churn rate · iOS users in Europe")

## 3. Run a proper A/B test on any metric and segment
Uses the same CUPED estimator as the main pipeline.

In [ ]:
def ab_test(metric="minutes", where="TRUE", arm="ai_feed"):
    d = q(f"SELECT arm, exp_{metric}_per_day AS y, pre_{metric}_per_day AS x FROM user_experiment WHERE {where}")
    t, c = d[d.arm == arm], d[d.arm == "control"]
    r = S.cuped(t.y.values, c.y.values, t.x.values, c.x.values)
    return pd.Series({"users_treatment": len(t), "users_control": len(c), "lift": f"{r['rel_lift']:+.2%}",
                      "95% CI": f"[{r['rel_ci_low']:+.2%}, {r['rel_ci_high']:+.2%}]", "p_value": r["p_value"],
                      "variance_removed_by_CUPED": f"{r['variance_reduction']:.0%}"})

ab_test("minutes", where="age_group = '55+'", arm="ai_feed")

In [ ]:
# metrics: minutes, active_days, sessions, likes, comments, dms_sent, shares, posts_created,
#          ad_revenue, friend_interactions, negative_feedback, ad_clicks, ad_impressions, stories_viewed
ab_test("friend_interactions", where="account_type = 'casual'", arm="ai_feed_social")

## 4. Time series from the raw event log

In [ ]:
daily = q('''
SELECT d.date, u.arm, SUM(d.minutes) / COUNT(DISTINCT u.user_id) AS minutes_per_active_user
FROM daily_activity d JOIN users u USING (user_id)
GROUP BY ALL ORDER BY d.date
''')
px.line(daily, x="date", y="minutes_per_active_user", color="arm", color_discrete_map=C.ARM_COLORS,
        title="Minutes per active user per day (experiment starts 2026-08-03)")

## 5. Plain pandas works too

In [ ]:
ux = pd.read_parquet(C.PROCESSED / "user_experiment.parquet")
ux.groupby(["region", "arm"]).churned.mean().unstack().style.format("{:.1%}").background_gradient(axis=None, cmap="Reds")